# HearSight Two-Stage Dataset Builder

Standalone Kaggle notebook. It reads the existing HearSight dataset, downloads S2TLD using Kaggle secrets when available, builds the two-stage detector/classifier dataset, audits it, and saves the output under `/kaggle/working`. No helper scripts are required.

In [ ]:
import os
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("RoboFlow_Key")
    secret_value_1 = user_secrets.get_secret("HF_TOKEN")
    os.environ["ROBOFLOW_API_KEY"] = secret_value_0
    os.environ["HF_TOKEN"] = secret_value_1
    print("Kaggle secrets loaded: RoboFlow_Key, HF_TOKEN")
except Exception as e:
    print("Kaggle secrets not available in this environment:", e)


In [ ]:
import importlib.util, subprocess, sys
packages = []
for module, package in [
    ("cv2", "opencv-python-headless"),
    ("yaml", "pyyaml"),
    ("roboflow", "roboflow"),
]:
    if importlib.util.find_spec(module) is None:
        packages.append(package)
if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


In [ ]:
"""Shared configuration for the HearSight two-stage training pipeline."""

from __future__ import annotations

SIGN_CLASSES = [
    "Bus stop",
    "Cross road",
    "Emergency exit",
    "Exit",
    "Fire alarm",
    "Fire extinguisher",
    "Men at work",
    "No entry",
    "No stopping or Standing",
    "Pedestrian crossing",
    "Pedestrian Prohibited",
    "Public toilet",
    "Railway crossing",
    "School ahead",
    "Speed breaker",
    "Stop",
    "Washroom (Female)",
    "Washroom (Male)",
    "Medical Shop - Hospital - First aid",
    # Additional signboard-only classes from new-classes.txt.
    "Side road left",
    "Side road right",
    "Narrow bridge ahead",
    "Narrow road ahead",
    "No parking",
    "Fuel station",
    "Roundabout",
    "Accessible / PwD",
    "Danger electricity",
    "Baggage claim",    
]

TRAFFIC_LIGHT_CLASSES = [
    "Traffic light red",
    "Traffic light yellow",
    "Traffic light green",
]

CLASSIFIER_CLASSES = SIGN_CLASSES + TRAFFIC_LIGHT_CLASSES + ["not_target"]

DETECTOR_CLASSES = [
    "road_sign",
    "facility_sign",
    "medical_sign",
    "traffic_light",
]

ROAD_SIGN_CLASSES = {
    "Bus stop",
    "Cross road",
    "Men at work",
    "No entry",
    "No stopping or Standing",
    "Pedestrian crossing",
    "Pedestrian Prohibited",
    "Railway crossing",
    "School ahead",
    "Speed breaker",
    "Stop",
    "Side road left",
    "Side road right",
    "Narrow bridge ahead",
    "Narrow road ahead",
    "No parking",
    "Fuel station",
    "Roundabout",
}

FACILITY_SIGN_CLASSES = {
    "Emergency exit",
    "Exit",
    "Fire alarm",
    "Fire extinguisher",
    "Public toilet",
    "Washroom (Female)",
    "Washroom (Male)",
    "Accessible / PwD",
    "Danger electricity",
    "Baggage claim",
}

MEDICAL_SIGN_CLASSES = {"Medical Shop - Hospital - First aid"}

GENERIC_CLASS_BY_SIGN_CLASS = {
    **{name: "road_sign" for name in ROAD_SIGN_CLASSES},
    **{name: "facility_sign" for name in FACILITY_SIGN_CLASSES},
    **{name: "medical_sign" for name in MEDICAL_SIGN_CLASSES},
}

S2TLD_CLASS_MAP = {
    "red": "Traffic light red",
    "yellow": "Traffic light yellow",
    "green": "Traffic light green",
}

S2TLD_NEGATIVE_LABELS = {"off", "wait on", "wait_on", "waiton"}

# Public Roboflow datasets selected in new-classes.txt and follow-up additions.
# Roboflow uses export format key `yolo26`; `yolov26` is not a valid Universe export key.
ROBOFLOW_DATASETS = [
    {
        "name": "indian_traffic_signboards",
        "workspace": "major-project-vu7ji",
        "project": "indian-traffic-signboards-zcmku",
        "version": 5,
        "license": "MIT",
        "classes": {
            "side road left": "Side road left",
            "side road right": "Side road right",
            "narrow bridge ahead": "Narrow bridge ahead",
            "narrow road ahead": "Narrow road ahead",
            "no parking": "No parking",
            "petrol pump ahead": "Fuel station",
            "roundabout": "Roundabout",
        },
    },
    {
        "name": "india_traffic_sign_roboroad",
        "workspace": "roboroad",
        "project": "india-traffic-sign-4x6if",
        "version": 2,
        "license": "CC BY 4.0",
        "classes": {
            "roundabout": "Roundabout",
        },
    },
    {
        "name": "bvi_signage_2",
        "workspace": "bvi",
        "project": "bvi-signage-2",
        "version": 6,
        "license": "CC BY 4.0",
        "classes": {
            "handicapped symbol": "Accessible / PwD",
            "danger-electricity": "Danger electricity",
        },
    },
    {
        "name": "airport_signage",
        "workspace": "bvi",
        "project": "airport_signage",
        "version": 7,
        "license": "CC BY 4.0",
        "classes": {
            "handicapped symbol": "Accessible / PwD",
            "person with a disability": "Accessible / PwD",
            "baggage claim": "Baggage claim",
        },
    },
    {
        "name": "bd_traffic_sign_detection_thesis",
        "workspace": "thesis-drlad",
        "project": "bd-traffic-sign-detection",
        "version": 1,
        "license": "Roboflow Universe",
        "classes": {
            "Filling Station Ahead": "Fuel station",
            "Narrow Bridge Ahead": "Narrow bridge ahead",
            "Narrow Road Ahead": "Narrow road ahead",
            "Road On Left": "Side road left",
            "Road On Right": "Side road right",
        },
    },
    {
        "name": "road_sign_detection_in_real_time_sit",
        "workspace": "sit-asmsw",
        "project": "road-sign-detection-in-real-time",
        "version": 4,
        "license": "CC BY 4.0",
        "classes": {
            "Petrol Pump/Gas Station": "Fuel station",
            "Petrol Pump/ Gas Station": "Fuel station",
            "Narrow Bridge": "Narrow bridge ahead",
            "Narrow road ahead": "Narrow road ahead",
            "Round About": "Roundabout",
            "No Parking": "No parking",
        },
    },
    {
        "name": "direction_signs",
        "workspace": "traffic-signs-dirwarn",
        "project": "direction-signs",
        "version": None,
        "license": "CC BY 4.0",
        "notes": "No downloadable Roboflow version was available when selected; used only if attached manually.",
        "classes": {
            "roundabout": "Roundabout",
        },
    },
    {
        "name": "road_sign_detection_in_bd_mostafinafis",
        "workspace": "mostafinafis",
        "project": "road-sign-detection-in-bd",
        "version": 1,
        "license": "Roboflow Universe",
        "classes": {
            "Petrol-Station-in-front": "Fuel station",
            "Side-road-left": "Side road left",
            "Side-road-right": "Side road right",
            "Ahead-of-the-bridge": "Narrow bridge ahead",
        },
    },
]


In [ ]:
#!/usr/bin/env python3
"""Build detector and classifier datasets for the HearSight two-stage system.

Outputs:
  output/
    detector_dataset/        YOLO detection dataset with generic classes
    classifier_dataset/      Ultralytics classification folder dataset
    manifests/               class maps and build summary

The detector is intentionally coarse. It proposes target-like signs and traffic
lights. The classifier is the precision gate and includes a not_target class.
"""

from __future__ import annotations

import argparse
import hashlib
import os
import json
import random
import re
import shutil
import urllib.request
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
from xml.etree import ElementTree as ET

import yaml


IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
ACTIVE_CLASSIFIER_CLASSES = list(CLASSIFIER_CLASSES)
ACTIVE_SIGN_CLASSES = list(SIGN_CLASSES)


@dataclass(frozen=True)
class Box:
    cls: str
    cx: float
    cy: float
    w: float
    h: float


def sanitize(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")


def file_md5(path: Path) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def ensure_empty(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def copy_image(src: Path, dst_dir: Path, prefix: str) -> Path:
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / f"{prefix}_{sanitize(src.stem)}{src.suffix.lower()}"
    i = 1
    while dst.exists():
        dst = dst_dir / f"{prefix}_{sanitize(src.stem)}_{i}{src.suffix.lower()}"
        i += 1
    shutil.copy2(src, dst)
    return dst


def find_image_for_label(label_path: Path, image_dir: Path) -> Path | None:
    for ext in IMAGE_EXTS:
        candidate = image_dir / f"{label_path.stem}{ext}"
        if candidate.exists():
            return candidate
        candidate = image_dir / f"{label_path.stem}{ext.upper()}"
        if candidate.exists():
            return candidate
    return None


def load_yaml_names(data_yaml: Path) -> list[str]:
    data = yaml.safe_load(data_yaml.read_text())
    names = data["names"]
    if isinstance(names, dict):
        return [names[i] for i in range(len(names))]
    return list(names)


def read_yolo_boxes(label_path: Path, class_names: list[str]) -> list[Box]:
    rows: list[Box] = []
    text = label_path.read_text().strip()
    if not text:
        return rows
    for line in text.splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls_id = int(float(parts[0]))
        if cls_id < 0 or cls_id >= len(class_names):
            continue
        rows.append(Box(class_names[cls_id], *(float(v) for v in parts[1:5])))
    return rows


def yolo_line(cls_id: int, box: Box) -> str:
    return f"{cls_id} {box.cx:.6f} {box.cy:.6f} {box.w:.6f} {box.h:.6f}"


def crop_box(
    image_path: Path,
    box: Box,
    dst_dir: Path,
    prefix: str,
    pad: float = 0.12,
    min_side: int = 48,
) -> Path | None:
    import cv2

    dst_dir.mkdir(parents=True, exist_ok=True)
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        return None
    height, width = img.shape[:2]
    bw, bh = box.w * width, box.h * height
    cx, cy = box.cx * width, box.cy * height
    half_w = max(bw * (0.5 + pad), min_side / 2)
    half_h = max(bh * (0.5 + pad), min_side / 2)
    x1 = max(0, int(cx - half_w))
    y1 = max(0, int(cy - half_h))
    x2 = min(width, int(cx + half_w))
    y2 = min(height, int(cy + half_h))
    if x2 <= x1 + 2 or y2 <= y1 + 2:
        return None
    crop = img[y1:y2, x1:x2]
    if min(crop.shape[:2]) < 10:
        return None
    dst = dst_dir / f"{prefix}_{sanitize(image_path.stem)}_{abs(hash((box.cls, box.cx, box.cy, box.w, box.h))) % 1000000}.jpg"
    cv2.imwrite(str(dst), crop, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    return dst


def random_not_target_crop(image_path: Path, dst_dir: Path, prefix: str) -> Path | None:
    import cv2

    dst_dir.mkdir(parents=True, exist_ok=True)
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        return None
    height, width = img.shape[:2]
    if width < 32 or height < 32:
        return None
    side = random.randint(max(32, min(width, height) // 5), max(33, min(width, height) // 2))
    x1 = random.randint(0, max(0, width - side))
    y1 = random.randint(0, max(0, height - side))
    crop = img[y1:min(height, y1 + side), x1:min(width, x1 + side)]
    dst = dst_dir / f"{prefix}_{sanitize(image_path.stem)}_{x1}_{y1}.jpg"
    cv2.imwrite(str(dst), crop, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    return dst


def prepare_source(source: Path, work_dir: Path) -> Path:
    if source.is_dir():
        return source
    if source.suffix.lower() != ".zip":
        raise ValueError(f"Unsupported source dataset: {source}")
    extract_dir = work_dir / "source_extracted"
    if not extract_dir.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(source) as zf:
            zf.extractall(extract_dir)
    data_yaml = next(extract_dir.rglob("data.yaml"), None)
    if data_yaml is None:
        raise FileNotFoundError(f"No data.yaml found inside {source}")
    return data_yaml.parent


def build_from_hearsight(source_root: Path, output: Path, seed: int, max_negatives_per_split: int) -> dict:
    random.seed(seed)
    names = load_yaml_names(source_root / "data.yaml")
    det_id = {name: i for i, name in enumerate(DETECTOR_CLASSES)}
    cls_counts = {split: {name: 0 for name in CLASSIFIER_CLASSES} for split in ("train", "val", "test")}
    det_counts = {split: {name: 0 for name in DETECTOR_CLASSES} for split in ("train", "val", "test")}
    neg_counts = {split: 0 for split in ("train", "val", "test")}
    seen_hash_split: dict[str, str] = {}
    duplicate_skips = 0

    for split in ("train", "val", "test"):
        image_dir = source_root / split / "images"
        label_dir = source_root / split / "labels"
        if not label_dir.exists():
            continue
        labels = sorted(label_dir.glob("*.txt"))
        random.shuffle(labels)
        split_negatives = 0
        for label_path in labels:
            image_path = find_image_for_label(label_path, image_dir)
            if image_path is None:
                continue
            md5 = file_md5(image_path)
            if md5 in seen_hash_split and seen_hash_split[md5] != split:
                duplicate_skips += 1
                continue
            seen_hash_split.setdefault(md5, split)

            boxes = read_yolo_boxes(label_path, names)
            kept_boxes = [b for b in boxes if b.cls in SIGN_CLASSES]
            dropped_boxes = [b for b in boxes if b.cls not in SIGN_CLASSES]
            det_lines: list[str] = []
            for box in kept_boxes:
                generic = GENERIC_CLASS_BY_SIGN_CLASS[box.cls]
                det_lines.append(yolo_line(det_id[generic], Box(generic, box.cx, box.cy, box.w, box.h)))
                det_counts[split][generic] += 1
                crop_box(image_path, box, output / "classifier_dataset" / split / sanitize(box.cls), split)
                cls_counts[split][box.cls] += 1

            for box in dropped_boxes:
                crop_box(image_path, box, output / "classifier_dataset" / split / "not_target", f"{split}_drop")
                cls_counts[split]["not_target"] += 1

            if not det_lines and split_negatives >= max_negatives_per_split:
                continue
            if not det_lines:
                split_negatives += 1
                neg_counts[split] += 1
                random_not_target_crop(image_path, output / "classifier_dataset" / split / "not_target", f"{split}_bg")
                cls_counts[split]["not_target"] += 1

            det_img = copy_image(image_path, output / "detector_dataset" / "images" / split, split)
            det_label = output / "detector_dataset" / "labels" / split / f"{det_img.stem}.txt"
            det_label.parent.mkdir(parents=True, exist_ok=True)
            det_label.write_text("\n".join(det_lines) + ("\n" if det_lines else ""))

    return {
        "hearsight_classifier_counts": cls_counts,
        "hearsight_detector_counts": det_counts,
        "hearsight_empty_detector_negatives": neg_counts,
        "cross_split_duplicate_skips": duplicate_skips,
    }


def normalize_class_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", name.lower()).strip()


def download_roboflow_dataset(spec: dict, work_dir: Path, api_key: str | None) -> Path | None:
    """Download one Roboflow Universe dataset via the official SDK.

    Direct Universe export URLs can return 403 in Kaggle even with a key. The
    SDK path matches hearsight-dataset-downloader.py and is more reliable.
    Prefer Roboflow's YOLO26 export key (`yolo26`), then fall back to yolov8/yolov7 if a project does not expose it.
    """
    if not api_key:
        print(f"[ROBOFLOW] skipping {spec['name']}: ROBOFLOW_API_KEY is not set")
        return None

    configured_version = spec.get("version")
    if configured_version is None:
        print(f"[ROBOFLOW] skipping SDK download for {spec['name']}: no downloadable version configured")
        return None

    work_dir.mkdir(parents=True, exist_ok=True)
    version_candidates = [int(configured_version)]
    for candidate in range(1, 16):
        if candidate not in version_candidates:
            version_candidates.append(candidate)

    try:
        from roboflow import Roboflow
        rf = Roboflow(api_key=api_key)
        workspace = rf.workspace(spec["workspace"])
        project = workspace.project(spec["project"])
        last_error = None

        for version_number in version_candidates:
            location = work_dir / f"{spec['name']}_v{version_number}"
            data_yaml = next(location.rglob("data.yaml"), None) if location.exists() else None
            if data_yaml is not None:
                print(f"[ROBOFLOW] using cached {spec['name']} v{version_number}: {data_yaml.parent}")
                return data_yaml.parent
            if location.exists():
                shutil.rmtree(location)

            try:
                version = project.version(version_number)
            except Exception as e:
                last_error = e
                if version_number == int(configured_version):
                    print(f"[ROBOFLOW] {spec['name']} v{version_number} is unavailable: {e}")
                continue

            print(
                f"[ROBOFLOW] downloading {spec['name']} "
                f"({spec['workspace']}/{spec['project']} v{version_number}) -> {location}"
            )
            for export_format in ("yolo26", "yolov8", "yolov7"):
                try:
                    dataset = version.download(export_format, location=str(location))
                    dataset_location = Path(dataset.location)
                    data_yaml = next(dataset_location.rglob("data.yaml"), None)
                    if data_yaml is None:
                        data_yaml = next(location.rglob("data.yaml"), None)
                    if data_yaml is None:
                        raise FileNotFoundError(f"No data.yaml found after {export_format} download")
                    print(f"[ROBOFLOW] downloaded {spec['name']} v{version_number} as {export_format}: {data_yaml.parent}")
                    return data_yaml.parent
                except Exception as e:
                    last_error = e
                    print(f"[ROBOFLOW] {spec['name']} v{version_number} {export_format} export failed: {e}")

        print(f"[ROBOFLOW] failed to download {spec['name']}: {last_error}")
        return None
    except Exception as e:
        print(f"[ROBOFLOW] failed to initialize/download {spec['name']}: {e}")
        return None


def discover_attached_roboflow_sources() -> dict[str, Path]:
    """Find manually attached Roboflow YOLO exports if SDK download fails.

    Do not treat the original HearSight dataset as a Roboflow source just because
    a few class names overlap. Prefer path/name evidence, then class evidence.
    """
    found: dict[str, Path] = {}
    roots = [Path("/kaggle/input"), Path("external_datasets"), Path(".")]
    data_yamls: list[Path] = []
    for root in roots:
        if root.exists():
            data_yamls.extend(root.glob("**/data.yaml"))
    for spec in ROBOFLOW_DATASETS:
        path_tokens = {
            spec["name"].lower(),
            spec["project"].lower(),
            spec["project"].replace("-", "_").lower(),
            spec["project"].replace("-", " ").lower(),
        }
        wanted = {normalize_class_name(k) for k in spec["classes"]}
        best: tuple[int, Path] | None = None
        for data_yaml in data_yamls:
            lower_path = str(data_yaml.parent).lower()
            normalized_path = lower_path.replace("_", " ").replace("-", " ")
            if "hearsight-35class" in lower_path or "final_dataset_combined" in lower_path:
                continue
            path_hits = sum(1 for token in path_tokens if token.replace("_", " ").replace("-", " ") in normalized_path)
            try:
                names = load_yaml_names(data_yaml)
                normalized_names = {normalize_class_name(n) for n in names}
                class_hits = len(wanted & normalized_names)
            except Exception:
                class_hits = 0
            # Require explicit path evidence, or at least two requested class hits from a non-HearSight source.
            if path_hits <= 0 and class_hits < 2:
                continue
            score = 10 * path_hits + 5 * class_hits
            if best is None or score > best[0]:
                best = (score, data_yaml.parent)
        if best is not None:
            found[spec["name"]] = best[1]
            print(f"[ROBOFLOW] attached fallback source for {spec['name']}: {best[1]}")
    return found


def build_from_external_yolo(
    source_root: Path,
    output: Path,
    source_name: str,
    class_map: dict[str, str],
    seed: int,
    max_images_per_target_class: int | None = None,
) -> dict:
    random.seed(seed)
    source_root = Path(source_root)
    data_yaml = source_root / "data.yaml"
    if not data_yaml.exists():
        data_yaml = next(source_root.rglob("data.yaml"), None)
        if data_yaml is None:
            return {f"{source_name}_used": False, f"{source_name}_reason": "missing data.yaml"}
        source_root = data_yaml.parent

    names = load_yaml_names(data_yaml)
    normalized_map = {normalize_class_name(src): dst for src, dst in class_map.items()}
    det_id = {name: i for i, name in enumerate(DETECTOR_CLASSES)}
    counts = {split: {name: 0 for name in CLASSIFIER_CLASSES} for split in ("train", "val", "test")}
    det_counts = {split: {name: 0 for name in DETECTOR_CLASSES} for split in ("train", "val", "test")}
    image_counts = {split: 0 for split in ("train", "val", "test")}
    capped_counts = {name: 0 for name in CLASSIFIER_CLASSES}
    skipped_unknown: dict[str, int] = {}

    split_candidates = [("train", "train"), ("valid", "val"), ("val", "val"), ("test", "test")]
    for src_split, dst_split in split_candidates:
        image_dir = source_root / src_split / "images"
        label_dir = source_root / src_split / "labels"
        if not label_dir.exists() or not image_dir.exists():
            continue
        labels = sorted(label_dir.glob("*.txt"))
        random.shuffle(labels)
        for label_path in labels:
            image_path = find_image_for_label(label_path, image_dir)
            if image_path is None:
                continue
            boxes = read_yolo_boxes(label_path, names)
            det_lines: list[str] = []
            kept_for_image = 0
            for box in boxes:
                target_cls = normalized_map.get(normalize_class_name(box.cls))
                if target_cls is None:
                    skipped_unknown[box.cls] = skipped_unknown.get(box.cls, 0) + 1
                    continue
                if target_cls not in GENERIC_CLASS_BY_SIGN_CLASS:
                    continue
                if max_images_per_target_class and capped_counts[target_cls] >= max_images_per_target_class:
                    continue
                generic = GENERIC_CLASS_BY_SIGN_CLASS[target_cls]
                det_lines.append(yolo_line(det_id[generic], Box(generic, box.cx, box.cy, box.w, box.h)))
                crop_box(
                    image_path,
                    Box(target_cls, box.cx, box.cy, box.w, box.h),
                    output / "classifier_dataset" / dst_split / sanitize(target_cls),
                    source_name,
                )
                counts[dst_split][target_cls] += 1
                det_counts[dst_split][generic] += 1
                capped_counts[target_cls] += 1
                kept_for_image += 1
            if not det_lines:
                continue
            det_img = copy_image(image_path, output / "detector_dataset" / "images" / dst_split, f"{source_name}_{dst_split}")
            det_label = output / "detector_dataset" / "labels" / dst_split / f"{det_img.stem}.txt"
            det_label.parent.mkdir(parents=True, exist_ok=True)
            det_label.write_text("\n".join(det_lines) + "\n")
            image_counts[dst_split] += 1

    return {
        f"{source_name}_used": True,
        f"{source_name}_source_root": str(source_root),
        f"{source_name}_image_counts": image_counts,
        f"{source_name}_classifier_counts": counts,
        f"{source_name}_detector_counts": det_counts,
        f"{source_name}_skipped_unknown_top20": dict(sorted(skipped_unknown.items(), key=lambda kv: kv[1], reverse=True)[:20]),
    }


def build_from_roboflow_sources(
    output: Path,
    seed: int,
    work_dir: Path,
    max_images_per_target_class: int | None = None,
) -> dict:
    api_key = os.environ.get("ROBOFLOW_API_KEY")
    attached = discover_attached_roboflow_sources()
    summary: dict = {"roboflow_sources": []}
    for spec in ROBOFLOW_DATASETS:
        source_root = download_roboflow_dataset(spec, work_dir, api_key)
        if source_root is None:
            source_root = attached.get(spec["name"])
        if source_root is None:
            summary["roboflow_sources"].append({"name": spec["name"], "used": False})
            continue
        source_summary = build_from_external_yolo(
            source_root=source_root,
            output=output,
            source_name=spec["name"],
            class_map=spec["classes"],
            seed=seed,
            max_images_per_target_class=max_images_per_target_class,
        )
        summary["roboflow_sources"].append({"name": spec["name"], "used": True, "root": str(source_root)})
        summary.update(source_summary)
    return summary


def iter_s2tld_annotations(zip_path: Path) -> Iterable[tuple[str, int, int, list[tuple[str, tuple[int, int, int, int]]]]]:
    with zipfile.ZipFile(zip_path) as zf:
        xml_files = [n for n in zf.namelist() if n.lower().endswith(".xml")]
        for xml_name in xml_files:
            root = ET.fromstring(zf.read(xml_name))
            filename = root.findtext("filename")
            width = int(root.findtext("size/width", "0"))
            height = int(root.findtext("size/height", "0"))
            objects = []
            for obj in root.findall("object"):
                label = (obj.findtext("name") or "").strip().lower()
                bb = obj.find("bndbox")
                if bb is None:
                    continue
                coords = tuple(int(float(bb.findtext(k, "0"))) for k in ("xmin", "ymin", "xmax", "ymax"))
                objects.append((label, coords))
            if filename and width > 0 and height > 0:
                yield filename, width, height, objects


def extract_s2tld_image(zip_path: Path, filename: str, dst_dir: Path, prefix: str) -> Path | None:
    with zipfile.ZipFile(zip_path) as zf:
        matches = [n for n in zf.namelist() if n.endswith(f"/JPEGImages/{filename}") or n.endswith(f"/{filename}")]
        matches = [n for n in matches if "/JPEGImages/" in n]
        if not matches:
            return None
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / f"{prefix}_{sanitize(Path(filename).stem)}.jpg"
        i = 1
        while dst.exists():
            dst = dst_dir / f"{prefix}_{sanitize(Path(filename).stem)}_{i}.jpg"
            i += 1
        with zf.open(matches[0]) as src, dst.open("wb") as out:
            shutil.copyfileobj(src, out)
        return dst




def _image_paths(root: Path) -> list[Path]:
    return sorted(p for p in root.glob("*") if p.suffix.lower() in IMAGE_EXTS)


def _read_image(path: Path):
    import cv2
    return cv2.imread(str(path), cv2.IMREAD_COLOR)


def _box_to_pixels(box: Box, width: int, height: int) -> tuple[int, int, int, int]:
    x1 = max(0, int((box.cx - box.w / 2) * width))
    y1 = max(0, int((box.cy - box.h / 2) * height))
    x2 = min(width, int((box.cx + box.w / 2) * width))
    y2 = min(height, int((box.cy + box.h / 2) * height))
    return x1, y1, x2, y2


def _collect_detector_objects(det_root: Path, split: str) -> tuple[list[tuple[str, Path, Box]], list[Path]]:
    names = DETECTOR_CLASSES
    objects: list[tuple[str, Path, Box]] = []
    backgrounds: list[Path] = []
    image_dir = det_root / "images" / split
    label_dir = det_root / "labels" / split
    for image_path in _image_paths(image_dir):
        label_path = label_dir / f"{image_path.stem}.txt"
        boxes = read_yolo_boxes(label_path, names) if label_path.exists() else []
        if not boxes:
            backgrounds.append(image_path)
        for box in boxes:
            objects.append((box.cls, image_path, box))
    if not backgrounds:
        backgrounds = _image_paths(image_dir)
    return objects, backgrounds


def _make_object_crop(image_path: Path, box: Box):
    import cv2
    img = _read_image(image_path)
    if img is None:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = _box_to_pixels(box, w, h)
    pad_x = int(max(2, (x2 - x1) * 0.12))
    pad_y = int(max(2, (y2 - y1) * 0.12))
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w, x2 + pad_x)
    y2 = min(h, y2 + pad_y)
    if x2 <= x1 + 3 or y2 <= y1 + 3:
        return None
    crop = img[y1:y2, x1:x2].copy()
    if min(crop.shape[:2]) < 6:
        return None
    return crop


def _jitter_crop(crop, rng: random.Random):
    import cv2
    import numpy as np
    out = crop.copy()
    if rng.random() < 0.80:
        alpha = rng.uniform(0.65, 1.25)
        beta = rng.randint(-35, 25)
        out = cv2.convertScaleAbs(out, alpha=alpha, beta=beta)
    if rng.random() < 0.35:
        k = rng.choice([3, 5])
        out = cv2.GaussianBlur(out, (k, k), 0)
    if rng.random() < 0.30:
        noise = np.random.default_rng(rng.randint(0, 2**31 - 1)).normal(0, rng.uniform(2, 8), out.shape)
        out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return out


def _paste_with_soft_edges(base, crop, x1: int, y1: int, rng: random.Random):
    import cv2
    import numpy as np
    h, w = crop.shape[:2]
    roi = base[y1:y1 + h, x1:x1 + w]
    if roi.shape[:2] != crop.shape[:2]:
        return
    mask = np.full((h, w, 1), rng.uniform(0.82, 1.0), dtype=np.float32)
    edge = max(1, min(h, w) // 12)
    if edge > 1:
        mask[:edge] *= np.linspace(0.35, 1.0, edge, dtype=np.float32).reshape(edge, 1, 1)
        mask[-edge:] *= np.linspace(1.0, 0.35, edge, dtype=np.float32).reshape(edge, 1, 1)
        mask[:, :edge] *= np.linspace(0.35, 1.0, edge, dtype=np.float32).reshape(1, edge, 1)
        mask[:, -edge:] *= np.linspace(1.0, 0.35, edge, dtype=np.float32).reshape(1, edge, 1)
    blended = crop.astype(np.float32) * mask + roi.astype(np.float32) * (1.0 - mask)
    base[y1:y1 + h, x1:x1 + w] = np.clip(blended, 0, 255).astype(np.uint8)


def synthesize_small_detector_objects(
    output: Path,
    seed: int = 42,
    train_images: int = 3000,
    val_images: int = 400,
    test_images: int = 400,
    max_objects_per_image: int = 4,
) -> dict:
    """Create detector-only hard samples with tiny/far pasted signs and traffic lights.

    Uses only objects/backgrounds from the same split to avoid train/val/test leakage.
    The classifier dataset is intentionally unchanged.
    """
    import cv2
    rng = random.Random(seed)
    det_root = output / "detector_dataset"
    det_id = {name: i for i, name in enumerate(DETECTOR_CLASSES)}
    requested = {"train": train_images, "val": val_images, "test": test_images}
    counts = {split: {name: 0 for name in DETECTOR_CLASSES} for split in requested}
    image_counts = {split: 0 for split in requested}
    skipped = {split: 0 for split in requested}
    area_ranges = [
        (0.0010, 0.0050),  # very far
        (0.0050, 0.0200),  # far
        (0.0200, 0.0800),  # medium
    ]

    for split, n_images in requested.items():
        objects, backgrounds = _collect_detector_objects(det_root, split)
        if not objects or not backgrounds or n_images <= 0:
            skipped[split] += n_images
            continue
        by_class: dict[str, list[tuple[str, Path, Box]]] = {name: [] for name in DETECTOR_CLASSES}
        for item in objects:
            by_class[item[0]].append(item)
        usable_classes = [name for name, items in by_class.items() if items]
        for idx in range(n_images):
            bg_path = rng.choice(backgrounds)
            base = _read_image(bg_path)
            if base is None:
                skipped[split] += 1
                continue
            h, w = base.shape[:2]
            if w < 160 or h < 160:
                skipped[split] += 1
                continue
            det_lines: list[str] = []
            n_obj = rng.randint(1, max_objects_per_image)
            for _ in range(n_obj):
                cls_name = rng.choice(usable_classes)
                item = rng.choice(by_class[cls_name])
                crop = _make_object_crop(item[1], item[2])
                if crop is None:
                    continue
                crop = _jitter_crop(crop, rng)
                area_low, area_high = rng.choices(area_ranges, weights=[0.50, 0.35, 0.15], k=1)[0]
                target_area = rng.uniform(area_low, area_high) * w * h
                aspect = crop.shape[1] / max(1, crop.shape[0])
                new_h = int(max(8, min(h * 0.30, (target_area / max(0.25, aspect)) ** 0.5)))
                new_w = int(max(8, min(w * 0.30, new_h * aspect)))
                if new_w >= w - 4 or new_h >= h - 4:
                    continue
                crop_resized = cv2.resize(crop, (new_w, new_h), interpolation=cv2.INTER_AREA)
                if rng.random() < 0.20 and min(new_w, new_h) > 12:
                    occ_w = rng.randint(2, max(3, new_w // 4))
                    occ_h = rng.randint(2, max(3, new_h // 4))
                    ox = rng.randint(0, max(0, new_w - occ_w))
                    oy = rng.randint(0, max(0, new_h - occ_h))
                    crop_resized[oy:oy + occ_h, ox:ox + occ_w] = crop_resized[max(0, oy - 1), max(0, ox - 1)].tolist()
                x1 = rng.randint(0, max(0, w - new_w - 1))
                y1 = rng.randint(0, max(0, h - new_h - 1))
                _paste_with_soft_edges(base, crop_resized, x1, y1, rng)
                cx = (x1 + new_w / 2) / w
                cy = (y1 + new_h / 2) / h
                bw = new_w / w
                bh = new_h / h
                det_lines.append(yolo_line(det_id[cls_name], Box(cls_name, cx, cy, bw, bh)))
                counts[split][cls_name] += 1
            if not det_lines:
                skipped[split] += 1
                continue
            out_img = det_root / "images" / split / f"synthetic_far_{split}_{idx:06d}.jpg"
            out_lbl = det_root / "labels" / split / f"synthetic_far_{split}_{idx:06d}.txt"
            cv2.imwrite(str(out_img), base, [int(cv2.IMWRITE_JPEG_QUALITY), rng.randint(72, 92)])
            out_lbl.write_text("\n".join(det_lines) + "\n")
            image_counts[split] += 1
    return {
        "synthetic_far_detector_images": image_counts,
        "synthetic_far_detector_counts": counts,
        "synthetic_far_detector_skipped": skipped,
    }

def build_from_s2tld(s2tld_zip: Path | None, output: Path, seed: int, max_images: int | None) -> dict:
    if s2tld_zip is None:
        return {"s2tld_used": False}
    random.seed(seed)
    annotations = list(iter_s2tld_annotations(s2tld_zip))
    buckets: dict[str, list[tuple[str, int, int, list[tuple[str, tuple[int, int, int, int]]]]]] = {}
    for item in annotations:
        labels = sorted({S2TLD_CLASS_MAP[label] for label, _ in item[3] if label in S2TLD_CLASS_MAP})
        bucket = labels[0] if labels else "not_target"
        buckets.setdefault(bucket, []).append(item)
    annotations = []
    for items in buckets.values():
        random.shuffle(items)
        annotations.extend(items)
    if max_images:
        annotations = annotations[:max_images]

    det_id = {name: i for i, name in enumerate(DETECTOR_CLASSES)}
    counts = {split: {name: 0 for name in TRAFFIC_NAMES_FOR_SUMMARY} for split in ("train", "val", "test")}
    split_by_filename: dict[str, str] = {}
    for items in buckets.values():
        for idx, item in enumerate(items):
            r = idx / max(1, len(items))
            split_by_filename[item[0]] = "train" if r < 0.80 else "val" if r < 0.90 else "test"

    for filename, width, height, objects in annotations:
        split = split_by_filename[filename]
        image_path = extract_s2tld_image(s2tld_zip, filename, output / "_tmp_s2tld_images", split)
        if image_path is None:
            continue
        det_lines = []
        for label, (xmin, ymin, xmax, ymax) in objects:
            cx = ((xmin + xmax) / 2) / width
            cy = ((ymin + ymax) / 2) / height
            bw = max(0.0, (xmax - xmin) / width)
            bh = max(0.0, (ymax - ymin) / height)
            if bw <= 0 or bh <= 0:
                continue
            box = Box("traffic_light", cx, cy, bw, bh)
            if label in S2TLD_CLASS_MAP:
                det_lines.append(yolo_line(det_id["traffic_light"], box))
                det_counts_name = S2TLD_CLASS_MAP[label]
                crop_box(image_path, Box(det_counts_name, cx, cy, bw, bh), output / "classifier_dataset" / split / sanitize(det_counts_name), "s2tld")
                counts[split][det_counts_name] += 1
            elif label in S2TLD_NEGATIVE_LABELS:
                crop_box(image_path, Box("not_target", cx, cy, bw, bh), output / "classifier_dataset" / split / "not_target", "s2tld_off")
                counts[split]["not_target"] += 1
        if det_lines:
            det_img = copy_image(image_path, output / "detector_dataset" / "images" / split, "s2tld")
            det_label = output / "detector_dataset" / "labels" / split / f"{det_img.stem}.txt"
            det_label.parent.mkdir(parents=True, exist_ok=True)
            det_label.write_text("\n".join(det_lines) + "\n")
    shutil.rmtree(output / "_tmp_s2tld_images", ignore_errors=True)
    return {"s2tld_used": True, "s2tld_counts": counts}


TRAFFIC_NAMES_FOR_SUMMARY = TRAFFIC_LIGHT_CLASSES = [
    "Traffic light red",
    "Traffic light yellow",
    "Traffic light green",
    "not_target",
]


def collect_nonempty_classifier_classes(output: Path) -> list[str]:
    active = [name for name in CLASSIFIER_CLASSES if name == "not_target"]
    for name in CLASSIFIER_CLASSES:
        if name == "not_target":
            continue
        class_dir_name = sanitize(name)
        has_images = False
        for split in ("train", "val", "test"):
            class_dir = output / "classifier_dataset" / split / class_dir_name
            if class_dir.exists() and any(p.suffix.lower() in IMAGE_EXTS for p in class_dir.iterdir()):
                has_images = True
                break
        if has_images:
            active.insert(-1, name)
    return active


def prune_empty_classifier_class_dirs(output: Path, active_classes: list[str]) -> dict:
    active_dirs = {sanitize(name) for name in active_classes}
    removed: dict[str, list[str]] = {split: [] for split in ("train", "val", "test")}
    for split in ("train", "val", "test"):
        root = output / "classifier_dataset" / split
        if not root.exists():
            continue
        for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            if class_dir.name not in active_dirs:
                shutil.rmtree(class_dir)
                removed[split].append(class_dir.name)
    return {"removed_empty_classifier_dirs": removed}


def remove_cross_split_duplicate_detector_images(output: Path) -> dict:
    """Remove duplicate detector images across splits, keeping train > val > test.

    Some external datasets contain identical images in multiple splits. The audit
    intentionally fails on this because it leaks validation/test information.
    """
    det_root = output / "detector_dataset"
    seen: dict[str, str] = {}
    removed = {split: 0 for split in ("train", "val", "test")}
    removed_files = {split: [] for split in ("train", "val", "test")}
    for split in ("train", "val", "test"):
        image_dir = det_root / "images" / split
        label_dir = det_root / "labels" / split
        if not image_dir.exists():
            continue
        for image_path in sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS):
            md5 = file_md5(image_path)
            if md5 not in seen:
                seen[md5] = split
                continue
            label_path = label_dir / f"{image_path.stem}.txt"
            image_path.unlink(missing_ok=True)
            label_path.unlink(missing_ok=True)
            removed[split] += 1
            if len(removed_files[split]) < 20:
                removed_files[split].append(str(image_path))
    return {
        "removed_cross_split_duplicate_detector_images": removed,
        "removed_cross_split_duplicate_detector_examples": removed_files,
    }


def finalize_active_classes(output: Path, drop_empty_classes: bool = True) -> dict:
    global ACTIVE_CLASSIFIER_CLASSES, ACTIVE_SIGN_CLASSES
    if drop_empty_classes:
        ACTIVE_CLASSIFIER_CLASSES = collect_nonempty_classifier_classes(output)
        ACTIVE_SIGN_CLASSES = [name for name in ACTIVE_CLASSIFIER_CLASSES if name not in TRAFFIC_LIGHT_CLASSES and name != "not_target"]
        summary = prune_empty_classifier_class_dirs(output, ACTIVE_CLASSIFIER_CLASSES)
    else:
        ACTIVE_CLASSIFIER_CLASSES = list(CLASSIFIER_CLASSES)
        ACTIVE_SIGN_CLASSES = list(SIGN_CLASSES)
        summary = {"removed_empty_classifier_dirs": {}}
    summary["active_sign_classes"] = ACTIVE_SIGN_CLASSES
    summary["active_classifier_classes"] = ACTIVE_CLASSIFIER_CLASSES
    summary["dropped_configured_classes"] = [name for name in CLASSIFIER_CLASSES if name not in ACTIVE_CLASSIFIER_CLASSES]
    return summary


def write_yaml(output: Path) -> None:
    det_yaml = {
        "path": ".",
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {i: name for i, name in enumerate(DETECTOR_CLASSES)},
    }
    (output / "detector_dataset" / "data.yaml").write_text(yaml.safe_dump(det_yaml, sort_keys=False))
    (output / "manifests").mkdir(parents=True, exist_ok=True)
    (output / "manifests" / "classifier_classes.json").write_text(json.dumps(ACTIVE_CLASSIFIER_CLASSES, indent=2))
    (output / "manifests" / "classifier_display_names.json").write_text(
        json.dumps({sanitize(name): name for name in ACTIVE_CLASSIFIER_CLASSES}, indent=2)
    )
    (output / "manifests" / "detector_classes.json").write_text(json.dumps(DETECTOR_CLASSES, indent=2))


def is_two_stage_dataset(path: Path) -> bool:
    path = Path(path)
    return (
        (path / "detector_dataset" / "data.yaml").exists()
        and (path / "detector_dataset" / "images").exists()
        and (path / "detector_dataset" / "labels").exists()
        and (path / "classifier_dataset").exists()
    )


def copy_two_stage_base_dataset(source: Path, output: Path) -> dict:
    """Preserve an existing two-stage dataset exactly as the build base."""
    source = Path(source)
    ensure_empty(output)
    shutil.copytree(source / "detector_dataset", output / "detector_dataset", dirs_exist_ok=True)
    shutil.copytree(source / "classifier_dataset", output / "classifier_dataset", dirs_exist_ok=True)
    if (source / "manifests").exists():
        shutil.copytree(source / "manifests", output / "manifests", dirs_exist_ok=True)
    else:
        (output / "manifests").mkdir(parents=True, exist_ok=True)
    # The copied data.yaml can contain an absolute or stale path. It is rewritten later by write_yaml().
    return {
        "base_dataset_mode": "two_stage_copy",
        "base_dataset_source": str(source),
    }



def create_dirs(output: Path) -> None:
    ensure_empty(output)
    for split in ("train", "val", "test"):
        (output / "detector_dataset" / "images" / split).mkdir(parents=True, exist_ok=True)
        (output / "detector_dataset" / "labels" / split).mkdir(parents=True, exist_ok=True)
        for name in ACTIVE_CLASSIFIER_CLASSES:
            (output / "classifier_dataset" / split / sanitize(name)).mkdir(parents=True, exist_ok=True)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--source", type=Path, default=Path("final_dataset_combined"))
    parser.add_argument("--output", type=Path, default=Path("hearsight_two_stage_dataset"))
    parser.add_argument("--work-dir", type=Path, default=Path(".hearsight_build_work"))
    parser.add_argument("--s2tld-zip", type=Path)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--max-negatives-per-split", type=int, default=2000)
    parser.add_argument("--max-s2tld-images", type=int)
    parser.add_argument("--roboflow-work-dir", type=Path, default=Path("/kaggle/working/roboflow_downloads"))
    parser.add_argument("--max-roboflow-images-per-class", type=int)
    parser.add_argument("--keep-empty-classes", action="store_true")
    parser.add_argument("--synthetic-train-images", type=int, default=3000)
    parser.add_argument("--synthetic-val-images", type=int, default=400)
    parser.add_argument("--synthetic-test-images", type=int, default=400)
    args = parser.parse_args()

    source_root = prepare_source(args.source, args.work_dir)
    if is_two_stage_dataset(source_root):
        summary = copy_two_stage_base_dataset(source_root, args.output)
    else:
        create_dirs(args.output)
        summary = build_from_hearsight(source_root, args.output, args.seed, args.max_negatives_per_split)
        summary.update(build_from_s2tld(args.s2tld_zip, args.output, args.seed, args.max_s2tld_images))
    summary.update(build_from_roboflow_sources(
        args.output,
        seed=args.seed,
        work_dir=args.roboflow_work_dir,
        max_images_per_target_class=args.max_roboflow_images_per_class,
    ))
    summary.update(synthesize_small_detector_objects(
        args.output,
        seed=args.seed,
        train_images=args.synthetic_train_images,
        val_images=args.synthetic_val_images,
        test_images=args.synthetic_test_images,
    ))
    summary.update(remove_cross_split_duplicate_detector_images(args.output))
    summary.update(finalize_active_classes(args.output, drop_empty_classes=not args.keep_empty_classes))
    write_yaml(args.output)
    (args.output / "manifests" / "build_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))


In [ ]:

from collections import Counter, defaultdict
from pathlib import Path
import hashlib, json, yaml

IMAGE_EXTS_AUDIT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def audit_two_stage_dataset(dataset):
    dataset = Path(dataset)
    det_root = dataset / "detector_dataset"
    cls_root = dataset / "classifier_dataset"
    data = yaml.safe_load((det_root / "data.yaml").read_text())
    detector_names = data["names"]
    detector_nc = len(detector_names)
    report = {
        "detector_classes": detector_names,
        "detector_counts": {},
        "empty_detector_labels": {},
        "bad_detector_labels": [],
        "classifier_counts": {},
        "cross_split_duplicate_hashes": 0,
    }
    def md5(path):
        h = hashlib.md5()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()
    hashes = defaultdict(set)
    for split in ("train", "val", "test"):
        det_counts = Counter()
        empty = 0
        for label_path in (det_root / "labels" / split).glob("*.txt"):
            text = label_path.read_text().strip()
            if not text:
                empty += 1
                continue
            for line in text.splitlines():
                cls_id = int(float(line.split()[0]))
                if cls_id < 0 or cls_id >= detector_nc:
                    report["bad_detector_labels"].append(str(label_path))
                det_counts[cls_id] += 1
        report["detector_counts"][split] = dict(det_counts)
        report["empty_detector_labels"][split] = empty
        cls_counts = {}
        for class_dir in sorted((cls_root / split).iterdir()):
            if class_dir.is_dir():
                cls_counts[class_dir.name] = sum(1 for p in class_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS_AUDIT)
        report["classifier_counts"][split] = cls_counts
        for image_path in (det_root / "images" / split).glob("*"):
            if image_path.suffix.lower() in IMAGE_EXTS_AUDIT:
                hashes[md5(image_path)].add(split)
    report["cross_split_duplicate_hashes"] = sum(1 for splits in hashes.values() if len(splits) > 1)
    return report


In [ ]:
from pathlib import Path
import urllib.request, os

S2TLD_URL = "https://huggingface.co/datasets/yangxue/S2TLD/resolve/main/S2TLD%EF%BC%881080x1920%EF%BC%89.zip"
S2TLD_ZIP = Path("/kaggle/working/S2TLD_1080x1920.zip")

if not S2TLD_ZIP.exists():
    headers = {}
    token = os.environ.get("HF_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"
    print("Downloading S2TLD...")
    req = urllib.request.Request(S2TLD_URL, headers=headers)
    with urllib.request.urlopen(req) as r, S2TLD_ZIP.open("wb") as f:
        f.write(r.read())
print("S2TLD:", S2TLD_ZIP, S2TLD_ZIP.stat().st_size)


In [ ]:
from pathlib import Path


def find_hearsight_source():
    candidates = [
        Path("/kaggle/input/datasets/ashok205/hearsight-two-stage-dataset-v2/hearsight_two_stage_dataset"),
        Path("/kaggle/input/hearsight-two-stage-dataset-v2/hearsight_two_stage_dataset"),
        Path("/kaggle/input/hearsight_two_stage_dataset-v2/hearsight_two_stage_dataset"),
        Path("hearsight_two_stage_dataset-v2"),
        # Older raw/source dataset fallback. Prefer the two-stage v2 dataset above.
        Path("/kaggle/input/datasets/ashok205/hearsight-35class/HearSight-dataset.zip"),
        Path("/kaggle/input/datasets/ashok205/hearsight-35class/final_dataset_combined"),
        Path("/kaggle/input/hearsight-35class/HearSight-dataset.zip"),
        Path("/kaggle/input/hearsight-35class/final_dataset_combined"),
    ]
    if Path("/kaggle/input").exists():
        candidates.extend(Path("/kaggle/input").glob("**/hearsight_two_stage_dataset"))
        candidates.extend(Path("/kaggle/input").glob("**/HearSight-dataset.zip"))
        candidates.extend(p.parent for p in Path("/kaggle/input").glob("**/data.yaml") if "hearsight" in str(p).lower())
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Attach hearsight-two-stage-dataset-v2, then rerun.")


SOURCE = find_hearsight_source()
OUTPUT = Path("/kaggle/working/hearsight_two_stage_dataset")
print("SOURCE =", SOURCE)
print("OUTPUT =", OUTPUT)
print("SOURCE_IS_TWO_STAGE =", is_two_stage_dataset(SOURCE))


In [ ]:
# Build the final two-stage dataset directly in this notebook.
# Prefer the existing HearSight two-stage v2 dataset and preserve its classes/samples.
# Then append S2TLD only when starting from a raw dataset, and append selected Roboflow signboard classes.
source_root = prepare_source(SOURCE, Path("/kaggle/working/.hearsight_build_work"))
if is_two_stage_dataset(source_root):
    summary = copy_two_stage_base_dataset(source_root, OUTPUT)
else:
    create_dirs(OUTPUT)
    summary = build_from_hearsight(source_root, OUTPUT, seed=42, max_negatives_per_split=2000)
    summary.update(build_from_s2tld(S2TLD_ZIP, OUTPUT, seed=42, max_images=None))
summary.update(build_from_roboflow_sources(
    OUTPUT,
    seed=42,
    work_dir=Path("/kaggle/working/roboflow_downloads"),
    max_images_per_target_class=None,
))
summary.update(synthesize_small_detector_objects(
    OUTPUT,
    seed=42,
    train_images=3000,
    val_images=400,
    test_images=400,
    max_objects_per_image=4,
))
summary.update(remove_cross_split_duplicate_detector_images(OUTPUT))
summary.update(finalize_active_classes(OUTPUT, drop_empty_classes=True))
write_yaml(OUTPUT)
(OUTPUT / "manifests" / "build_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2)[:12000])


In [ ]:
report = audit_two_stage_dataset(OUTPUT)
print(json.dumps(report, indent=2)[:12000])
assert report["cross_split_duplicate_hashes"] == 0
assert not report["bad_detector_labels"]


In [ ]:
import shutil
zip_base = "/kaggle/working/hearsight_two_stage_dataset"
shutil.make_archive(zip_base, "zip", "/kaggle/working", "hearsight_two_stage_dataset")
print("Wrote", zip_base + ".zip")
print("Publish /kaggle/working/hearsight_two_stage_dataset as your new Kaggle dataset.")
